# S5.1 · 五级基线阶梯

本项目的科学核心不是「联邦学习能不能跑通」，而是**联邦学习相对于合规成本低得多的替代方案，究竟多带来多少价值**。

| 级别 | 含义 | 跨境暴露面 |
|---|---|---|
| L0 | 内地单方建模 | 无 |
| L1 | 加对方 k-匿名聚合统计 | 聚合量，非个人信息 |
| L2 | 加对方粗粒度标记 | 逐人低维标记 |
| L3 | 纵向联邦（LR / GBDT / SplitNN） | 中间量逐人交换 |
| L4 | 集中式（原始数据汇集） | 全量个人信息 |

**L1 是 VFL 真正的竞争者**：它几乎没有合规成本。如果 L1 能拿到大部分价值，本项目的商业前提就不成立。

In [1]:
ROUND_DP = 4          # 表格展示精度（不影响任何计算结果）
import sys, subprocess, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "registry").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, yaml
CONFIG_PATH = ROOT / "modules/m5_modeling/configs/experiment.yaml"
config = yaml.safe_load(open(CONFIG_PATH, encoding="utf-8"))
seed = config.get("seeds", [config.get("seed")])[0]
git = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, cwd=ROOT).stdout.strip()
print("config:", CONFIG_PATH.relative_to(ROOT))
print("seed  :", seed, "| 全部种子:", config.get("seeds"))
print("git   :", git or "(未提交)")
print("numpy :", np.__version__, "| pandas:", pd.__version__)

config: modules/m5_modeling/configs/experiment.yaml
seed  : 11 | 全部种子: [11, 22, 33, 44, 55]
git   : 9f0643d
numpy : 2.3.5 | pandas: 2.3.3


In [2]:
import time
from modules.m2_synthetic.components.scm_generator import load_scenarios
from modules.m5_modeling.components.experiment import run_one
scen_cfg = yaml.safe_load(open(ROOT / 'modules/m2_synthetic/configs/scenarios.yaml', encoding='utf-8'))
scenarios = load_scenarios(scen_cfg)
hp, seeds = config['hyperparams'], config['seeds']
t0 = time.time(); rows = []
for c in scenarios:
    for sd in seeds:
        for sp in config['splits']:
            rows += run_one(c, sd, hp, sp)
df = pd.DataFrame(rows)
df.to_csv(ROOT / 'modules/m5_modeling/results/ladder_results_raw.csv', index=False)
print(f'网格完成 {time.time()-t0:.0f}s | {len(df)} 行 | {df.scenario.nunique()} 场景 × {df.seed.nunique()} 种子 × {df.split.nunique()} 划分')

网格完成 43s | 1440 行 | 8 场景 × 5 种子 × 2 划分


## 表1 · 阶梯总表（全场景合并，随机划分，均值 ± 95% CI）

置信区间由**跨种子**变异给出——框架要求 ≥5 种子且主指标带区间，此项不降级。

In [3]:
CI_Z = 1.96
core = df[df.auc.notna()].copy()
def ci95(x):
    x = np.asarray(x, float); m = x.mean()
    se = x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0
    return pd.Series({'AUC均值': m, 'CI下界': m - CI_Z*se, 'CI上界': m + CI_Z*se})
core[core.split == 'random'].groupby('level').auc.apply(ci95).unstack().sort_values('AUC均值', ascending=False).round(ROUND_DP)

,AUC均值,CI下界,CI上界
level,,,
L4_集中式_LR,0.8003,0.7759,0.8248
L3a_联邦LR,0.7978,0.7724,0.8232
L3b_纵向GBDT,0.7697,0.7437,0.7958
L4_集中式_GBDT,0.7697,0.7437,0.7958
L3c_SplitNN_形态A双向,0.7664,0.7345,0.7982
L3c_SplitNN_形态B自监督,0.7213,0.6954,0.7472
L2_加粗粒度标记_LR,0.7183,0.7001,0.7364
L0_内地单方_LR,0.7127,0.6964,0.7290
L1_加k匿名统计_LR,0.7097,0.6929,0.7265


## 表2 · 划分口径：D-1 随机 vs D-2 时间外推（OOT）

金融业的真实口径是 OOT。差值为正说明随机划分**高估**了性能。

In [4]:
p = core.pivot_table(index='level', columns='split', values='auc', aggfunc='mean')
p['随机减OOT'] = p['random'] - p['oot']
p.sort_values('随机减OOT', ascending=False).round(ROUND_DP)

split,oot,random,随机减OOT
level,,,
L3c_SplitNN_形态B自监督,0.7084,0.7213,0.0129
L3c_SplitNN_形态B随机,0.6992,0.7076,0.0084
L3b_纵向GBDT,0.7643,0.7697,0.0054
L4_集中式_GBDT,0.7643,0.7697,0.0054
L0_内地单方_GBDT,0.6726,0.6758,0.0032
L0_内地单方_LR,0.7101,0.7127,0.0026
L2_加粗粒度标记_LR,0.7164,0.7183,0.0019
L1_加k匿名统计_LR,0.7082,0.7097,0.0015
L4_集中式_LR,0.8011,0.8003,-0.0008


## 表3 · 逐场景：L3a 联邦LR 相对 L1 的净增益

In [5]:
w = core[core.split == 'random'].pivot_table(index=['scenario','seed'], columns='level', values='auc')
g = (w['L3a_联邦LR'] - w['L1_加k匿名统计_LR']).groupby('scenario').apply(ci95).unstack()
g['显著'] = np.where(g['CI下界'] > 0, '是', '否')
g.round(ROUND_DP)

,AUC均值,CI下界,CI上界,显著
scenario,,,,
S1_基准,0.0800,0.0534,0.1066,是
S2_零互补,0.0278,0.0103,0.0452,是
S3_高互补,0.2074,0.1733,0.2416,是
S4_高冗余,0.0685,0.0574,0.0797,是
S5_低重叠高漂移,0.0701,-0.0155,0.1557,否
S6_匹配噪声,0.0733,0.0413,0.1052,是
S7_同意选择偏差,0.0619,0.0301,0.0937,是
S8_稀疏正样本,0.1153,0.0481,0.1826,是


## 表4 · 通信成本

In [6]:
c = core[core.split=='random'].groupby('level')[['comm_rounds','comm_floats']].mean().dropna(how='all')
c[c.comm_floats > 0].round(0)

,comm_rounds,comm_floats
level,,
L1_加k匿名统计_LR,NaN,382.0
L3a_联邦LR,400.0,3540940.0
L3b_纵向GBDT,60.0,1748319.0
L3c_SplitNN_形态A双向,200.0,408576.0
L3c_SplitNN_形态B自监督,200.0,204288.0
L3c_SplitNN_形态B随机,200.0,204288.0
L4_集中式_GBDT,60.0,1748319.0


## 表5 · 决策等价性：L1 与 L3 的 Top-10% 名单重合度

AUC 接近**不等于**决策一致。真正影响业务的是名单换了多少人。

In [7]:
o = df[df['topk_overlap@10'].notna()]
o[o.split=='random'].groupby('level')['topk_overlap@10'].agg(['mean','std']).round(ROUND_DP)

,mean,std
level,,
OVERLAP_L1_vs_L3a_联邦LR,0.4593,0.1088
OVERLAP_L1_vs_L3b_纵向GBDT,0.3666,0.0903
OVERLAP_L1_vs_L3c_SplitNN_形态A双向,0.3868,0.0877


## 表6 · 增量（uplift）评估

**响应率 ≠ 增量**。营销真正要的是「因为营销才转化」的人。

In [8]:
u = df[df.auuc.notna()]
u[u.split=='random'].groupby('level')[['auuc','uplift@10']].mean().round(ROUND_DP)

,auuc,uplift@10
level,,
L0_内地单方_UPLIFT,0.0059,0.0557
L1_加k匿名统计_UPLIFT,0.0058,0.0504
L2_加粗粒度标记_UPLIFT,0.0059,0.0616
L4_集中式_UPLIFT,0.0076,0.1066


## 表7 · 稳健性检验：调参能否救回 L1？

上面的全部结论都用**同一组固定超参**。一个合理的质疑是：「L1 只捕获 12.3%」会不会只是 L1 欠调参？

依 DR-GOV-009 的红线「L1 必须与 L3 同等认真实现」，**L1 的搜索空间刻意给到最大**——60 组，是 L0 的 12 倍。若调参能翻转 C1，必须让它有机会翻转。

方法上有一条硬要求：**选参只看验证集**（60/20/20 三分）。在测试集上选参会让搜索空间大的级别虚高更多，污染方向恰好**有利于 L1**，同样会毁掉结论。

In [9]:
hs = pd.read_csv(ROOT / 'modules/m5_modeling/results/hyperparam_search.csv')
t = hs.groupby('level').agg(网格点数=('n_points','first'),
                            验证AUC=('valid_auc','mean'),
                            测试AUC=('test_auc','mean'))
t['验证减测试'] = t['验证AUC'] - t['测试AUC']
t.sort_values('测试AUC', ascending=False).round(ROUND_DP)

,网格点数,验证AUC,测试AUC,验证减测试
level,,,,
L4_LR,5,0.8460,0.8390,0.0070
L3a_联邦LR,27,0.8466,0.8352,0.0114
L3c_SplitNN,54,0.8451,0.8210,0.0241
L3b_纵向GBDT,81,0.8366,0.7954,0.0412
L2_LR,5,0.6989,0.7254,-0.0265
L1_LR,60,0.7265,0.7148,0.0116
L0_LR,5,0.6945,0.7122,-0.0177


「验证减测试」一列随网格增大而增大（L3b 的 81 组差 0.0412），说明选参乐观度被正确检出——这是方法本身的自检。

### 最强的一击：让 L1 直接在测试集上挑最优点

下表的 oracle 上界**不是有效估计**（它用了测试集选参，等于作弊），但它给出「调参最多能帮 L1 到什么程度」的上限。

In [10]:
ob = pd.read_csv(ROOT / 'modules/m5_modeling/results/oracle_upper_bound.csv')
o = ob.groupby('level').oracle_test_auc.mean()
h = hs.groupby('level').test_auc.mean()
cmp2 = pd.DataFrame({'正规选参': h[o.index], 'oracle上界': o})
cmp2['乐观量'] = cmp2['oracle上界'] - cmp2['正规选参']
cmp2.round(ROUND_DP)

,正规选参,oracle上界,乐观量
level,,,
L0_LR,0.7122,0.7158,0.0036
L1_LR,0.7148,0.7351,0.0203
L3a_联邦LR,0.8352,0.8404,0.0052


In [11]:
PCT = 100
for tag, col in [('正规选参', h), ('oracle上界（作弊）', o)]:
    gap_l1 = col['L1_LR'] - col['L0_LR']
    gap_l3 = col['L3a_联邦LR'] - col['L0_LR']
    print(f'{tag:18s} L1−L0={gap_l1:+.4f}  L3a−L0={gap_l3:+.4f}  '
          f'L1 捕获 {gap_l1/gap_l3*PCT:.1f}%')

正规选参               L1−L0=+0.0026  L3a−L0=+0.1230  L1 捕获 2.1%
oracle上界（作弊）       L1−L0=+0.0193  L3a−L0=+0.1246  L1 捕获 15.5%


**C1 站住了，而且更强了。**

- 正规口径：L1 只捕获 **2.1%**
- oracle 口径（让 L1 作弊）：也只有 **15.5%**
- 此前 40 种子安慰剂对照给出 12.3%，正好落在两者之间

三个独立口径互相印证：**调参救不回 L1**。L1 的瓶颈是结构性的——分段键只能用主动方特征，回传的统计量因而是主动方特征的函数。